In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
path="/content/drive/MyDrive/BGCA-master2/cross_domain_depression"
os.chdir(path)
os.listdir(path)

['run.ipynb',
 'times.ttf',
 'timesbd.ttf',
 'requirements.txt',
 'Untitled0.ipynb',
 'predict.py',
 'run_training.sh',
 'data_preparation.py',
 'domain_prompt_builder.py',
 'uncertainty_regularizer.py',
 'evaluator.py',
 'main.py',
 'models',
 'data',
 'utils',
 '__pycache__',
 'data_loader.py',
 'Qwen',
 'llm_adapter.py',
 'config.py',
 'trainer.py',
 'test_result_laptop_reddit.json',
 'test_result_laptop_twitter.json',
 'bin',
 'train.py',
 'results_kd_meta.json',
 'predictions_log_stage_2.json',
 'teacher_model_1.pt',
 'teacher_model_2.pt',
 'teacher_model_3.pt',
 'teacher_model_4.pt',
 't5-small',
 'baselines',
 'student_stage1_source1.pt',
 'student_stage2_source1.pt',
 'tsne_laptop_to_depression.pdf',
 'tsne_laptop_to_depression.svg',
 'features_for_tsne.pkl',
 'student_stage1_source0.pt',
 'student_stage2_source0.pt',
 'tsne_reloaded.svg',
 'tsne_reloaded.pdf',
 'tsne_ideal_alignment.pdf',
 'tsne_ideal_alignment.svg',
 'tsne_natural_alignment.pdf',
 'tsne_natural_alignment.svg'

In [ ]:
# 1. 完整训练
!python train2.py --mode kd_meta

=========================kd_meta================================
TRAINING WITH KNOWLEDGE DISTILLATION + META-LEARNING
RESUMING FROM PHASE 2
tokenizer_config.json: 2.54kB [00:00, 8.96MB/s]
spiece.model: 100% 792k/792k [00:00<00:00, 1.44MB/s]
tokenizer.json: 2.42MB [00:00, 68.8MB/s]
special_tokens_map.json: 2.20kB [00:00, 9.28MB/s]
config.json: 100% 662/662 [00:00<00:00, 4.51MB/s]
model.safetensors: 100% 3.13G/3.13G [00:09<00:00, 333MB/s]
Loading weights: 100% 558/558 [00:00<00:00, 790.95it/s, Materializing param=shared.weight]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
generation_config.json: 100% 147/147 [00:00<00:00, 818kB/s]
Loading weights: 100% 131/131 [00:02<00:00, 48.39it/s, Materializing param=shared.weight]
Teacher model:google/flan-t5-large,Teacher params:783150080
Stude

In [ ]:
!python t5.py

=====================t5-large==============================
Loading weights: 100% 509/509 [00:00<00:00, 917.38it/s, Materializing param=shared.weight]
Loaded 3265 samples from ./data/reddit/train.txt
Loaded 156 samples from ./data/reddit/dev.txt
Loaded 66 samples from ./data/reddit/test.txt

Epoch 1/10
Training: 100% 409/409 [06:04<00:00,  1.12it/s]
Train Loss: 0.8495
Evaluating: 100% 20/20 [00:22<00:00,  1.12s/it]
Validation - Accuracy: 0.2687, Precision: 0.2951, Recall: 0.2719, F1: 0.2824
Model saved to ./best_t5_depression.pt

Epoch 2/10
Training: 100% 409/409 [06:04<00:00,  1.12it/s]
Train Loss: 0.3084
Evaluating: 100% 20/20 [00:26<00:00,  1.33s/it]
Validation - Accuracy: 0.4313, Precision: 0.4770, Recall: 0.4943, F1: 0.4837
Model saved to ./best_t5_depression.pt

Epoch 3/10
Training: 100% 409/409 [06:04<00:00,  1.12it/s]
Train Loss: 0.2375
Evaluating: 100% 20/20 [00:27<00:00,  1.37s/it]
Validation - Accuracy: 0.5125, Precision: 0.5620, Recall: 0.5687, F1: 0.5643
Model saved to ./b

In [ ]:
# 2. 只评估
!python train.py --mode test --test data/twitter/test.txt

In [ ]:
# 3. 交互式测试
!python train.py --mode interactive


In [ ]:
# 4. 消融实验
!python train.py --mode ablation

In [ ]:
### 数据集统计
def statistics(domain,data_type):
  nums = 0
  data = open(f'data/{domain}/{data_type}','r',encoding='utf-8').readlines()
  for line in data:
    triplets = eval(line.strip().split("####")[1])
    nums += len(triplets)
  return nums

print(statistics('twitter','train.txt') + statistics('twitter','test.txt') + statistics('twitter','dev.txt'))
print(statistics('reddit','train.txt') + statistics('reddit','test.txt') + statistics('reddit','dev.txt'))

In [ ]:
### 将Rest15和reset16提取三元组
def extract_triplets(domain,data_type):
  new_data = []
  data = open(f"data/{domain}/quad/{data_type}",'r',encoding='utf-8').readlines()
  for line in data:
    sentence,triplets = line.strip().split("####")
    t = []
    for aspect,category,sentiment,opinion in eval(triplets):
      ##去掉方面词
      cate = "#".join([s.upper() for s in category.strip().split()])
      t.append((cate,opinion,sentiment))
    # print(sentence + "####" + f"{t}")
    new_data.append(sentence + "####" + f"{t}")

  ## 保存新的三元组文件
  ##
  with open(f"data/{domain}/{data_type}",'a',encoding='utf-8') as files:
    for l in new_data:
      files.write(l + "\n")
  print(f"data/{domain}/{data_type} is saved successfully")
  return

domains = ['rest15','rest16']
data_types = ['train.txt','test.txt','dev.txt']
for domain in domains:
  for data_type in data_types:
    extract_triplets(domain,data_type)


In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
import os

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
import os

class DepressionTripletDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = []

        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if '####' in line:
                    text, triplets_str = line.strip().split('####')
                    triplets = eval(triplets_str)
                    self.data.append((text.strip(), triplets))

        print(f"Loaded {len(self.data)} samples from {file_path}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, triplets = self.data[idx]
        input_text = text
        target_text = " | ".join([f"({t[0]}, {t[1]}, {t[2]})" for t in triplets])

        input_encoding = self.tokenizer(input_text,
                                        max_length=self.max_length,
                                        padding='max_length',
                                        truncation=True,
                                        return_tensors='pt')

        target_encoding = self.tokenizer(target_text,
                                         max_length=128,
                                         padding='max_length',
                                         truncation=True,
                                         return_tensors='pt')

        labels = target_encoding['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': labels,
            'raw_text': text,
            'raw_triplets': triplets
        }

def train_epoch(model, data_loader, optimizer, device):
    model.train()
    losses = []
    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return sum(losses) / len(losses)

@torch.no_grad()
def evaluate(model, data_loader, tokenizer, device):
    model.eval()
    preds = []
    targets = []
    for batch in tqdm(data_loader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )
        pred_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        target_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

        preds.extend(pred_texts)
        targets.extend(target_texts)

    acc = accuracy_score(targets, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(targets, preds, average='micro')
    return acc, precision, recall, f1

def custom_collate_fn(batch):
    """处理变长序列的batch整理"""
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    labels = torch.stack([item['labels'] for item in batch])
    sentiment_labels = torch.tensor([item['sentiment_label'] for item in batch])

    raw_texts = [item['raw_text'] for item in batch]
    raw_triplets = [item['raw_triplets'] for item in batch]

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
        'sentiment_label': sentiment_labels,
        'raw_text': raw_texts,
        'raw_triplets': raw_triplets
    }



def save_checkpoint(model, path):
    torch.save(model.state_dict(), path)
    print(f"Model saved to {path}")

def load_checkpoint(model, path, device):
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    print(f"Model loaded from {path}")

def main():
    train_file = "./data/twitter/train.txt"
    val_file = "./data/twitter/dev.txt"
    test_file = "./data/twitter/test.txt"

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = T5Tokenizer.from_pretrained('t5-base')
    model = T5ForConditionalGeneration.from_pretrained('t5-base').to(device)

    train_dataset = DepressionTripletDataset(train_file, tokenizer)
    val_dataset = DepressionTripletDataset(val_file, tokenizer)
    test_dataset = DepressionTripletDataset(test_file, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,collate_fn=custom_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=8,shuffle=True,collate_fn=custom_collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=8,shuffle=True,collate_fn=custom_collate_fn)

    optimizer = optim.AdamW(model.parameters(), lr=5e-5)

    epochs = 20
    patience = 3      # 连续多少个epoch验证集F1未提升就早停
    best_f1 = 0.0
    patience_counter = 0
    checkpoint_path = "./best_t5_depression.pt"

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        train_loss = train_epoch(model, train_loader, optimizer, device)
        print(f"Train Loss: {train_loss:.4f}")

        acc, precision, recall, f1 = evaluate(model, val_loader, tokenizer, device)
        print(f"Validation - Accuracy: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            save_checkpoint(model, checkpoint_path)
        else:
            patience_counter += 1
            print(f"No improvement in F1 for {patience_counter} epochs")

        if patience_counter >= patience:
            print(f"Early stopping triggered after {patience_counter} epochs without improvement.")
            break

    # 加载最优模型并在测试集评估
    load_checkpoint(model, checkpoint_path, device)
    acc, precision, recall, f1 = evaluate(model, test_loader, tokenizer, device)
    print(f"\nTest Set - Accuracy: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

if __name__ == '__main__':
    main()
